# MPG-FER Final One-Shot — Kaggle T4

**Motif–Pixel Graph Network for FER2013**  
Final engineering candidate after re-reviewing the H1 / CRS / Issue #90 lineage.

This notebook is intentionally **self-contained** and avoids PyG/DGL so it can run on a stock Kaggle GPU image without a fragile binary install.

## Locked scientific/engineering decisions

1. **Pixel stays explicit**: 48×48 = **2304 pixel nodes**.
2. **H1 relational signal is retained continuously**: each pixel receives a 5×5 center-relative descriptor (24 normalized relative differences + local scale), coordinates, intensity and first-order geometry.
3. **No hard K128/K512/Q_M21 quantization**: the negative representation bottleneck is not reintroduced.
4. **Pixel graph is real message passing**: explicit fixed 8-neighbor adjacency; sender/receiver attention is edge-conditioned by `dx,dy,distance,ΔI,|ΔI|`.
5. **Motif is an explicit middle representation**: after local pixel-graph reasoning, 49 overlapping 12×12 spatial supports create learned **motif occurrence nodes**.
6. **Motif type is soft and trainable**: a learnable motif prototype bank provides a type distribution; no argmax, no stability threshold, no connected-component bottleneck.
7. **Geometry is explicit at the motif graph**: 49 motif nodes form a complete graph with relative `dx,dy,distance,direction` attention bias.
8. **No CNN backbone** and no image-level bypass. Both the pixel branch and motif branch originate from graph processing.
9. **PublicTest (`val.csv`) selects the checkpoint**. PrivateTest (`test.csv`) is evaluated only after the best validation checkpoint is fixed.
10. **Final metric uses a predeclared 2-view TTA**: original + horizontal flip. Raw metrics are also reported.

### Final dataflow

```text
48×48 grayscale
    ↓
2304 pixel nodes
    ↓
continuous H1-inspired 5×5 relational features
    ↓
8-neighbor edge-aware Pixel Graph Attention ×4
    ↓
96-D local structural embeddings
    ├──────────────────────────────→ pixel graph readout (128-D)
    ↓
Spatial Motif Composer
- 48 learned motif types
- soft assignment
- 49 overlapping 12×12 / stride-6 motif occurrences
- WHAT + TYPE + WHERE
    ↓
49 motif nodes, 192-D
    ↓
complete geometry-aware Motif Graph Transformer ×5
    ↓
motif graph readout (384-D)
    ↓
concat 128 + 384 = 512
    ↓
MLP 512→256→7
```

The motif branch receives stronger auxiliary supervision than the pixel branch so motif cannot become a decorative layer, while the pixel readout prevents another irreversible bottleneck.

> **No accuracy is guaranteed in advance.** The notebook is designed to remove implementation/infrastructure ambiguity and make the single full run scientifically interpretable.


## 1. User Configuration

In [ ]:
from pathlib import Path

# -------------------------
# Dataset / Kaggle
# -------------------------
KAGGLE_DATASET_SLUG = "doduyquynii/fer13-split"
KNOWN_DATA_ROOTS = [
    Path("/kaggle/input/datasets/doduyquynii/fer13-split/fer13-split"),
    Path("/kaggle/input/fer13-split/fer13-split"),
    Path("/kaggle/input/fer13-split"),
]
FALLBACK_DOWNLOAD_ROOT = Path("/kaggle/working/fer13_data")

# train.csv = Train, val.csv = PublicTest, test.csv = PrivateTest
EXPECTED_COUNTS = {"train": 28709, "val": 3589, "test": 3589}

# -------------------------
# Output
# -------------------------
OUTPUT_ROOT = Path("/kaggle/working/outputs/mpg_fer_final_one_shot_seed42")
CHECKPOINT_PATH = OUTPUT_ROOT / "best_val_acc.pt"
LAST_PATH = OUTPUT_ROOT / "last.pt"
ARCHIVE_PATH = Path("/kaggle/working/mpg_fer_final_one_shot_seed42.zip")

# -------------------------
# Reproducibility / training
# -------------------------
SEED = 42
RUN_MICRO_OVERFIT = True
RUN_FULL_TRAINING = True
MAX_EPOCHS = 80
MIN_EPOCHS = 25
EARLY_STOP_PATIENCE = 15

BATCH_SIZE = 16           # chosen conservatively for a Kaggle T4 16 GB
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2      # effective batch ≈ 32
NUM_WORKERS = 2

BASE_LR = 3e-4
MIN_LR_RATIO = 0.05
WARMUP_EPOCHS = 5
WEIGHT_DECAY = 5e-4
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.05

# Auxiliary branch supervision / motif anti-collapse.
MOTIF_CE_WEIGHT = 0.20
PIXEL_CE_WEIGHT = 0.05
MOTIF_DIVERSITY_WEIGHT = 0.02
MOTIF_BALANCE_WEIGHT = 0.002

# -------------------------
# Model
# -------------------------
PIXEL_DIM = 96
PIXEL_HEADS = 4
PIXEL_BLOCKS = 4

MOTIF_TYPES = 48
MOTIF_DIM = 192
MOTIF_HEADS = 8
MOTIF_BLOCKS = 5
MOTIF_WINDOW = 12
MOTIF_STRIDE = 6

DROPOUT = 0.10
CLASSIFIER_DROPOUT = 0.35
NUM_CLASSES = 7

# Predeclared final inference protocol.
USE_TTA_FOR_SELECTION = True  # original + horizontal flip on PublicTest
USE_TTA_FOR_PRIVATE = True

assert SEED == 42
assert MOTIF_WINDOW == 12 and MOTIF_STRIDE == 6
assert 48 % 1 == 0 and PIXEL_DIM % PIXEL_HEADS == 0 and MOTIF_DIM % MOTIF_HEADS == 0
assert CHECKPOINT_PATH.parent == OUTPUT_ROOT
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## 2. Kaggle API Secret + Dataset Resolution

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def configure_kaggle_api():
    """Configure ~/.kaggle/kaggle.json from Kaggle Secrets without printing secrets."""
    username = None
    key = None
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        try:
            username = client.get_secret("KAGGLE_USERNAME")
        except Exception:
            pass
        try:
            key = client.get_secret("KAGGLE_KEY")
        except Exception:
            pass
    except Exception:
        pass

    username = username or os.environ.get("KAGGLE_USERNAME")
    key = key or os.environ.get("KAGGLE_KEY")

    if username and key:
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        credential_path = kaggle_dir / "kaggle.json"
        credential_path.write_text(
            json.dumps({"username": username, "key": key}),
            encoding="utf-8",
        )
        os.chmod(credential_path, 0o600)
        print("Kaggle API: configured from secrets (values hidden).")
        return True

    print("Kaggle API: secret not found. Mounted Kaggle Inputs will still work.")
    return False

KAGGLE_API_READY = configure_kaggle_api()

def is_split_root(path: Path):
    return path.is_dir() and all((path / f"{s}.csv").is_file() for s in ("train", "val", "test"))

def resolve_data_root():
    for candidate in KNOWN_DATA_ROOTS:
        if is_split_root(candidate):
            return candidate

    # Search Kaggle inputs without assuming one mount convention.
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for train_csv in input_root.rglob("train.csv"):
            candidate = train_csv.parent
            if is_split_root(candidate):
                return candidate

    # API fallback: only used when the dataset was not mounted.
    if not KAGGLE_API_READY:
        raise FileNotFoundError(
            "FER split dataset is not mounted and Kaggle API secrets are unavailable."
        )

    if FALLBACK_DOWNLOAD_ROOT.exists():
        shutil.rmtree(FALLBACK_DOWNLOAD_ROOT)
    FALLBACK_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable, "-m", "kaggle", "datasets", "download",
        "-d", KAGGLE_DATASET_SLUG,
        "-p", str(FALLBACK_DOWNLOAD_ROOT),
        "--unzip",
    ]
    print("$", " ".join(command))
    subprocess.run(command, check=True)

    if is_split_root(FALLBACK_DOWNLOAD_ROOT):
        return FALLBACK_DOWNLOAD_ROOT
    for train_csv in FALLBACK_DOWNLOAD_ROOT.rglob("train.csv"):
        candidate = train_csv.parent
        if is_split_root(candidate):
            return candidate
    raise FileNotFoundError("Downloaded dataset does not contain train.csv/val.csv/test.csv together.")

DATA_ROOT = resolve_data_root()
SPLIT_PATHS = {s: DATA_ROOT / f"{s}.csv" for s in ("train", "val", "test")}
print("DATA_ROOT:", DATA_ROOT)
for split, path in SPLIT_PATHS.items():
    print(split, "->", path)


## 3. Environment Inspection and Seed

In [ ]:
import importlib.metadata
import platform
import random
import time
import numpy as np
import pandas as pd
import torch
import torchvision
import sklearn

print("python:", sys.version)
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("This notebook is locked for Kaggle GPU/T4. Enable a GPU accelerator before running.")

DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Reproducibility first; custom graph ops here do not need cuDNN convolutions.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "seed": SEED,
}
(OUTPUT_ROOT / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")


## 4. Load and Strictly Validate FER2013 Splits

In [ ]:
from collections import Counter

def read_fer_csv(path: Path, expected_count: int):
    df = pd.read_csv(path)
    lowered = {c.lower().strip(): c for c in df.columns}
    if "emotion" not in lowered or "pixels" not in lowered:
        raise RuntimeError(f"{path}: expected columns 'emotion' and 'pixels', got {list(df.columns)}")
    emo_col = lowered["emotion"]
    pix_col = lowered["pixels"]

    if len(df) != expected_count:
        raise RuntimeError(f"{path}: row count {len(df)} != expected {expected_count}")

    labels = df[emo_col].to_numpy(dtype=np.int64)
    if labels.min() < 0 or labels.max() >= NUM_CLASSES:
        raise RuntimeError(f"{path}: label range {labels.min()}..{labels.max()} is invalid")

    images = np.empty((len(df), 48, 48), dtype=np.uint8)
    for i, text in enumerate(df[pix_col].astype(str).tolist()):
        arr = np.fromstring(text, sep=" ", dtype=np.uint8)
        if arr.size != 48 * 48:
            raise RuntimeError(f"{path}: row {i} has {arr.size} pixels, expected 2304")
        images[i] = arr.reshape(48, 48)

    return images, labels

split_arrays = {}
for split in ("train", "val", "test"):
    print("Loading", split)
    split_arrays[split] = read_fer_csv(SPLIT_PATHS[split], EXPECTED_COUNTS[split])
    images, labels = split_arrays[split]
    print(split, images.shape, images.dtype, Counter(labels.tolist()))

train_images, train_labels = split_arrays["train"]
val_images, val_labels = split_arrays["val"]
test_images, test_labels = split_arrays["test"]

TRAIN_MEAN = float(train_images.mean() / 255.0)
TRAIN_STD = float(train_images.std() / 255.0)
if TRAIN_STD < 1e-3:
    raise RuntimeError("Train image std is unexpectedly small.")

print("train mean/std [0,1]:", TRAIN_MEAN, TRAIN_STD)

# Public/Private evaluation protocol is explicit and recorded.
dataset_manifest = {
    "data_root": str(DATA_ROOT),
    "counts": EXPECTED_COUNTS,
    "train_mean": TRAIN_MEAN,
    "train_std": TRAIN_STD,
    "semantics": {"train": "Train", "val": "PublicTest", "test": "PrivateTest"},
}
(OUTPUT_ROOT / "dataset_manifest.json").write_text(json.dumps(dataset_manifest, indent=2), encoding="utf-8")


## 5. Dataset and Conservative FER Augmentation

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TVF
from torchvision.transforms import InterpolationMode

class FERDataset(Dataset):
    def __init__(self, images, labels, augment=False, fill_value=0.5):
        self.images = images
        self.labels = labels
        self.augment = augment
        self.fill_value = float(fill_value)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = torch.from_numpy(self.images[index]).float().unsqueeze(0) / 255.0
        label = int(self.labels[index])

        if self.augment:
            if random.random() < 0.5:
                image = torch.flip(image, dims=[2])

            # Mild affine augmentation. It changes the image BEFORE graph construction;
            # the graph itself remains the same 48x48 pixel topology.
            angle = random.uniform(-10.0, 10.0)
            tx = random.randint(-3, 3)
            ty = random.randint(-3, 3)
            scale = random.uniform(0.95, 1.05)
            image = TVF.affine(
                image,
                angle=angle,
                translate=[tx, ty],
                scale=scale,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
                fill=[self.fill_value],
            )

            # Very mild photometric perturbation; preserve grayscale facial structure.
            if random.random() < 0.25:
                image = TVF.adjust_brightness(image, random.uniform(0.90, 1.10))
            if random.random() < 0.25:
                image = TVF.adjust_contrast(image, random.uniform(0.90, 1.10))
            image = image.clamp(0.0, 1.0)

        return image, label

train_dataset = FERDataset(train_images, train_labels, augment=True, fill_value=TRAIN_MEAN)
train_eval_dataset = FERDataset(train_images, train_labels, augment=False, fill_value=TRAIN_MEAN)
val_dataset = FERDataset(val_images, val_labels, augment=False, fill_value=TRAIN_MEAN)
test_dataset = FERDataset(test_images, test_labels, augment=False, fill_value=TRAIN_MEAN)

def worker_init_fn(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

def make_loader(dataset, batch_size, shuffle, workers=NUM_WORKERS):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=workers,
        pin_memory=True,
        persistent_workers=(workers > 0),
        worker_init_fn=worker_init_fn,
        generator=loader_generator if shuffle else None,
        drop_last=False,
    )


## 6. Explicit Pixel Graph / Relational Buffers

In [ ]:
import math

def _reflect_index(i, n):
    # Equivalent to reflect padding for radius 2.
    if n <= 1:
        return 0
    while i < 0 or i >= n:
        if i < 0:
            i = -i
        if i >= n:
            i = 2 * n - 2 - i
    return i

def build_graph_buffers(height=48, width=48, motif_window=12, motif_stride=6):
    directions = [
        (-1, -1), (-1, 0), (-1, 1),
        ( 0, -1),           ( 0, 1),
        ( 1, -1), ( 1, 0),  ( 1, 1),
    ]

    n_nodes = height * width
    neighbor_index = np.zeros((n_nodes, 8), dtype=np.int64)
    neighbor_valid = np.zeros((n_nodes, 8), dtype=np.bool_)
    neighbor_rel = np.zeros((n_nodes, 8, 3), dtype=np.float32)

    for r in range(height):
        for c in range(width):
            i = r * width + c
            for k, (dr, dc) in enumerate(directions):
                rr, cc = r + dr, c + dc
                neighbor_rel[i, k] = [dc, dr, math.sqrt(dc * dc + dr * dr)]
                if 0 <= rr < height and 0 <= cc < width:
                    neighbor_index[i, k] = rr * width + cc
                    neighbor_valid[i, k] = True
                else:
                    neighbor_index[i, k] = i  # ignored by mask

    # H1-inspired 5x5 center-relative support: the 24 non-center offsets.
    offsets_5x5 = [
        (dr, dc)
        for dr in range(-2, 3)
        for dc in range(-2, 3)
        if not (dr == 0 and dc == 0)
    ]
    rel5_index = np.zeros((n_nodes, 24), dtype=np.int64)
    for r in range(height):
        for c in range(width):
            i = r * width + c
            for k, (dr, dc) in enumerate(offsets_5x5):
                rr = _reflect_index(r + dr, height)
                cc = _reflect_index(c + dc, width)
                rel5_index[i, k] = rr * width + cc

    ys = np.linspace(-1.0, 1.0, height, dtype=np.float32)
    xs = np.linspace(-1.0, 1.0, width, dtype=np.float32)
    coords = np.stack(np.meshgrid(xs, ys), axis=-1).reshape(-1, 2)

    # 12x12 windows, stride 6 -> 7x7 = 49 overlapping motif-occurrence supports.
    windows = []
    for r in range(0, height - motif_window + 1, motif_stride):
        for c in range(0, width - motif_window + 1, motif_stride):
            windows.append([
                (r + dr) * width + (c + dc)
                for dr in range(motif_window)
                for dc in range(motif_window)
            ])

    buffers = {
        "neighbor_index": torch.from_numpy(neighbor_index),
        "neighbor_valid": torch.from_numpy(neighbor_valid),
        "neighbor_rel": torch.from_numpy(neighbor_rel),
        "rel5_index": torch.from_numpy(rel5_index),
        "coords": torch.from_numpy(coords),
        "window_index": torch.tensor(windows, dtype=torch.long),
    }
    return buffers

GRAPH_BUFFERS = build_graph_buffers(
    height=48, width=48,
    motif_window=MOTIF_WINDOW,
    motif_stride=MOTIF_STRIDE,
)

print({k: tuple(v.shape) for k, v in GRAPH_BUFFERS.items()})
assert GRAPH_BUFFERS["neighbor_index"].shape == (2304, 8)
assert GRAPH_BUFFERS["rel5_index"].shape == (2304, 24)
assert GRAPH_BUFFERS["window_index"].shape == (49, 144)


## 7. Pixel Relational Encoder + Edge-aware Pixel GNN

Important: the implementation below does **not** use a convolutional backbone. The 5×5 feature builder is deterministic relational feature extraction. The learnable local processor is explicit neighbor message passing on the 8-neighbor graph.

The 5×5 descriptor keeps the core H1 idea but removes PCA/KMeans as irreversible bottlenecks:

- 24 center-relative differences
- local scale normalization with `eps = 1/255`
- `log(local_sigma)`
- intensity + `(x,y)`
- `gx, gy, |grad|, laplacian`

The first learned linear layer replaces the old fixed PCA projection and can adapt end-to-end to FER.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class PixelFeatureEncoder(nn.Module):
    def __init__(self, buffers, train_mean, train_std, eps=1/255):
        super().__init__()
        for key in ("neighbor_index", "neighbor_valid", "neighbor_rel", "rel5_index", "coords"):
            self.register_buffer(key, buffers[key].clone(), persistent=False)
        self.register_buffer("train_mean", torch.tensor(float(train_mean)), persistent=True)
        self.register_buffer("train_std", torch.tensor(float(train_std)), persistent=True)
        self.eps = float(eps)
        self.out_dim = 32

        # direction order: NW,N,NE,W,E,SW,S,SE
        self.idx_n = 1
        self.idx_w = 3
        self.idx_e = 4
        self.idx_s = 6

    def forward(self, images):
        # images: [B,1,48,48], raw [0,1]
        b = images.shape[0]
        raw = images[:, 0].reshape(b, -1)
        center = raw

        rel5_values = raw[:, self.rel5_index]            # [B,2304,24]
        patch25 = torch.cat([center.unsqueeze(-1), rel5_values], dim=-1)
        local_sigma = patch25.std(dim=-1, unbiased=False)
        denom = torch.sqrt(local_sigma.square() + self.eps ** 2)
        rel5 = (rel5_values - center.unsqueeze(-1)) / denom.unsqueeze(-1)
        log_sigma = torch.log(local_sigma + self.eps).unsqueeze(-1)

        neighbor_values = raw[:, self.neighbor_index]    # [B,2304,8]
        gx = 0.5 * (neighbor_values[:, :, self.idx_e] - neighbor_values[:, :, self.idx_w])
        gy = 0.5 * (neighbor_values[:, :, self.idx_s] - neighbor_values[:, :, self.idx_n])
        grad = torch.sqrt(gx.square() + gy.square() + 1e-8)
        lap = (
            neighbor_values[:, :, self.idx_n]
            + neighbor_values[:, :, self.idx_s]
            + neighbor_values[:, :, self.idx_w]
            + neighbor_values[:, :, self.idx_e]
            - 4.0 * center
        )

        intensity = ((center - self.train_mean) / self.train_std.clamp_min(1e-6)).unsqueeze(-1)
        coords = self.coords.unsqueeze(0).expand(b, -1, -1)

        node_features = torch.cat([
            intensity,
            coords,
            rel5,
            log_sigma,
            gx.unsqueeze(-1),
            gy.unsqueeze(-1),
            grad.unsqueeze(-1),
            lap.unsqueeze(-1),
        ], dim=-1)

        fixed_geometry = self.neighbor_rel.unsqueeze(0).expand(b, -1, -1, -1)
        delta_i = (neighbor_values - center.unsqueeze(-1)).unsqueeze(-1)
        edge_features = torch.cat([
            fixed_geometry,
            delta_i,
            delta_i.abs(),
        ], dim=-1)  # [B,2304,8,5]

        return node_features, edge_features

class PixelGraphBlock(nn.Module):
    def __init__(self, dim, heads=4, edge_dim=5, mlp_ratio=3.0, dropout=0.1):
        super().__init__()
        assert dim % heads == 0
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads

        self.norm1 = nn.LayerNorm(dim)
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)

        self.edge_bias = nn.Sequential(
            nn.Linear(edge_dim, 32),
            nn.GELU(),
            nn.Linear(32, heads),
        )
        self.edge_value = nn.Sequential(
            nn.Linear(edge_dim, 32),
            nn.GELU(),
            nn.Linear(32, dim),
        )

        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

        hidden = int(dim * mlp_ratio)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, neighbor_index, neighbor_valid, edge_features):
        b, n, d = x.shape
        k_neighbors = neighbor_index.shape[1]

        z = self.norm1(x)
        q = self.q(z).view(b, n, self.heads, self.head_dim)
        k_all = self.k(z).view(b, n, self.heads, self.head_dim)
        v_all = self.v(z).view(b, n, self.heads, self.head_dim)

        # Explicit graph gather: each receiver i sees exactly its listed 8-neighbor senders j.
        k_n = k_all[:, neighbor_index]   # [B,N,8,H,Dh]
        v_n = v_all[:, neighbor_index]

        edge_bias = self.edge_bias(edge_features)  # [B,N,8,H]
        edge_value = self.edge_value(edge_features).view(
            b, n, k_neighbors, self.heads, self.head_dim
        )

        score = (
            (q.unsqueeze(2) * k_n).sum(dim=-1) / math.sqrt(self.head_dim)
            + edge_bias
        )
        mask = neighbor_valid.unsqueeze(0).unsqueeze(-1)
        score = score.masked_fill(~mask, -1e4)
        attention = score.softmax(dim=2)

        aggregate = (
            attention.unsqueeze(-1) * (v_n + edge_value)
        ).sum(dim=2).reshape(b, n, d)

        x = x + self.drop(self.proj(aggregate))
        x = x + self.ffn(self.norm2(x))
        return x

class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = max(32, dim // 2)
        self.score = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        weights = self.score(x).squeeze(-1).softmax(dim=1)
        return (weights.unsqueeze(-1) * x).sum(dim=1)


## 8. Spatial Motif Composer

Why this version instead of unconstrained global slots / TopK:

- one motif node is an **occurrence**, not a motif type;
- 49 overlapping windows preserve location and prevent a type from merging eyebrow/mouth occurrences into one node;
- every pixel participates in one or more candidate occurrences;
- soft motif type assignment is trainable;
- no discrete `argmax`, connected-component extraction, or node dropping.

Each motif node explicitly contains:

- **WHAT**: attentive local structural embedding,
- **TYPE**: soft distribution over 48 learned motif prototypes,
- **WHERE**: learned center, spread and confidence.


In [ ]:
class SpatialMotifComposer(nn.Module):
    def __init__(
        self,
        pixel_dim,
        motif_dim,
        num_types,
        buffers,
        type_dim=48,
        dropout=0.1,
        temperature=0.25,
    ):
        super().__init__()
        self.pixel_dim = pixel_dim
        self.num_types = num_types
        self.temperature = float(temperature)

        self.register_buffer("window_index", buffers["window_index"].clone(), persistent=False)
        self.register_buffer("coords", buffers["coords"].clone(), persistent=False)

        self.prototypes = nn.Parameter(torch.empty(num_types, pixel_dim))
        # Orthogonal start gives the prototype bank maximal initial diversity.
        nn.init.orthogonal_(self.prototypes)

        self.saliency = nn.Sequential(
            nn.LayerNorm(pixel_dim),
            nn.Linear(pixel_dim, pixel_dim // 2),
            nn.GELU(),
            nn.Linear(pixel_dim // 2, 1),
        )
        self.type_projection = nn.Sequential(
            nn.Linear(num_types, type_dim),
            nn.GELU(),
        )

        input_dim = pixel_dim + type_dim + 5
        self.output = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, motif_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(motif_dim, motif_dim),
        )

    def forward(self, pixel_embeddings):
        b, n, d = pixel_embeddings.shape

        normalized_pixels = F.normalize(pixel_embeddings, dim=-1)
        normalized_prototypes = F.normalize(self.prototypes, dim=-1)

        type_logits = torch.einsum(
            "bnd,md->bnm",
            normalized_pixels,
            normalized_prototypes,
        ) / self.temperature
        assignment = type_logits.softmax(dim=-1)   # [B,2304,M]

        idx = self.window_index
        local_h = pixel_embeddings[:, idx]         # [B,49,144,D]
        local_a = assignment[:, idx]               # [B,49,144,M]

        # Saliency chooses the important pixels within each local occurrence support.
        # Prototype confidence is only a weak prior, not a hard gate.
        confidence = local_a.max(dim=-1).values
        saliency_logits = (
            self.saliency(local_h).squeeze(-1)
            + 0.25 * torch.log(confidence.clamp_min(1e-6))
        )
        weights = saliency_logits.softmax(dim=-1)

        what = (weights.unsqueeze(-1) * local_h).sum(dim=2)
        type_distribution = (weights.unsqueeze(-1) * local_a).sum(dim=2)
        type_embedding = self.type_projection(type_distribution)

        local_coords = self.coords[idx].unsqueeze(0).to(pixel_embeddings.dtype)
        position = (weights.unsqueeze(-1) * local_coords).sum(dim=2)
        spread = torch.sqrt(
            (
                weights.unsqueeze(-1)
                * (local_coords - position.unsqueeze(2)).square()
            ).sum(dim=2)
            + 1e-6
        )
        motif_confidence = (weights * confidence).sum(dim=2, keepdim=True)

        where = torch.cat([position, spread, motif_confidence], dim=-1)
        motif_nodes = self.output(torch.cat([what, type_embedding, where], dim=-1))

        # Mild anti-collapse losses. They are deliberately small and do not define
        # the motif semantics; FER supervision remains the main objective.
        gram = normalized_prototypes @ normalized_prototypes.T
        eye = torch.eye(self.num_types, device=gram.device, dtype=gram.dtype)
        diversity_loss = (
            (gram - eye).square().sum()
            / (self.num_types * (self.num_types - 1))
        )

        usage = assignment.mean(dim=(0, 1)).clamp_min(1e-8)
        balance_loss = (
            usage * (usage.log() + math.log(self.num_types))
        ).sum()  # KL(mean_usage || uniform)

        aux = {
            "motif_diversity": diversity_loss,
            "motif_balance": balance_loss,
            "motif_usage_min": usage.min().detach(),
            "motif_usage_max": usage.max().detach(),
        }
        return motif_nodes, position, type_distribution, aux


## 9. Geometry-aware Complete Motif Graph + Full MPG-FER

In [ ]:
class GeometryGraphBlock(nn.Module):
    def __init__(self, dim, heads=8, geometry_dim=7, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        assert dim % heads == 0
        self.dim = dim
        self.heads = heads
        self.head_dim = dim // heads

        self.norm1 = nn.LayerNorm(dim)
        self.qkv = nn.Linear(dim, dim * 3, bias=False)

        self.geometry_bias = nn.Sequential(
            nn.Linear(geometry_dim, 64),
            nn.GELU(),
            nn.Linear(64, heads),
        )

        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

        hidden = int(dim * mlp_ratio)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, position):
        # Complete graph attention. N=49, so dense pairwise geometry is cheap.
        b, n, d = x.shape
        z = self.norm1(x)

        q, k, v = self.qkv(z).chunk(3, dim=-1)
        q = q.view(b, n, self.heads, self.head_dim).transpose(1, 2)
        k = k.view(b, n, self.heads, self.head_dim).transpose(1, 2)
        v = v.view(b, n, self.heads, self.head_dim).transpose(1, 2)

        score = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)

        # delta[i,j] = position_j - position_i
        delta = position[:, None, :, :] - position[:, :, None, :]
        dx = delta[..., 0]
        dy = delta[..., 1]
        distance = torch.sqrt(dx.square() + dy.square() + 1e-8)
        ux = dx / (distance + 1e-6)
        uy = dy / (distance + 1e-6)

        geometry = torch.stack([
            dx,
            dy,
            distance,
            distance.square(),
            ux,
            uy,
            1.0 / (1.0 + distance),
        ], dim=-1)

        bias = self.geometry_bias(geometry).permute(0, 3, 1, 2)
        attention = (score + bias).softmax(dim=-1)

        aggregate = torch.matmul(attention, v)
        aggregate = aggregate.transpose(1, 2).reshape(b, n, d)

        x = x + self.drop(self.proj(aggregate))
        x = x + self.ffn(self.norm2(x))
        return x

class MPGFER(nn.Module):
    def __init__(self, buffers, train_mean, train_std):
        super().__init__()

        self.feature_encoder = PixelFeatureEncoder(
            buffers=buffers,
            train_mean=train_mean,
            train_std=train_std,
        )

        for key in ("neighbor_index", "neighbor_valid"):
            self.register_buffer(key, buffers[key].clone(), persistent=False)

        self.input_projection = nn.Sequential(
            nn.Linear(self.feature_encoder.out_dim, PIXEL_DIM),
            nn.LayerNorm(PIXEL_DIM),
            nn.GELU(),
        )

        self.pixel_blocks = nn.ModuleList([
            PixelGraphBlock(
                dim=PIXEL_DIM,
                heads=PIXEL_HEADS,
                dropout=DROPOUT,
            )
            for _ in range(PIXEL_BLOCKS)
        ])

        self.pixel_attention_pool = AttentionPool(PIXEL_DIM)
        self.pixel_readout = nn.Sequential(
            nn.Linear(PIXEL_DIM * 3, 128),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

        self.motif_composer = SpatialMotifComposer(
            pixel_dim=PIXEL_DIM,
            motif_dim=MOTIF_DIM,
            num_types=MOTIF_TYPES,
            buffers=buffers,
            type_dim=48,
            dropout=DROPOUT,
            temperature=0.25,
        )

        self.motif_blocks = nn.ModuleList([
            GeometryGraphBlock(
                dim=MOTIF_DIM,
                heads=MOTIF_HEADS,
                dropout=DROPOUT,
            )
            for _ in range(MOTIF_BLOCKS)
        ])

        self.motif_attention_pool = AttentionPool(MOTIF_DIM)
        self.motif_readout = nn.Sequential(
            nn.Linear(MOTIF_DIM * 3, 384),
            nn.GELU(),
            nn.Dropout(DROPOUT),
        )

        # Deep supervision: motif branch gets the larger weight.
        self.pixel_aux_head = nn.Linear(128, NUM_CLASSES)
        self.motif_aux_head = nn.Linear(384, NUM_CLASSES)

        self.classifier = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(CLASSIFIER_DROPOUT),
            nn.Linear(256, NUM_CLASSES),
        )

    def forward(self, images, return_aux=False):
        node_features, edge_features = self.feature_encoder(images)
        x = self.input_projection(node_features)

        for block in self.pixel_blocks:
            x = block(
                x,
                neighbor_index=self.neighbor_index,
                neighbor_valid=self.neighbor_valid,
                edge_features=edge_features,
            )

        pixel_vector = torch.cat([
            x.mean(dim=1),
            x.amax(dim=1),
            self.pixel_attention_pool(x),
        ], dim=-1)
        pixel_vector = self.pixel_readout(pixel_vector)

        motif_nodes, motif_position, motif_type, motif_aux = self.motif_composer(x)

        for block in self.motif_blocks:
            motif_nodes = block(motif_nodes, motif_position)

        motif_vector = torch.cat([
            motif_nodes.mean(dim=1),
            motif_nodes.amax(dim=1),
            self.motif_attention_pool(motif_nodes),
        ], dim=-1)
        motif_vector = self.motif_readout(motif_vector)

        logits = self.classifier(torch.cat([pixel_vector, motif_vector], dim=-1))

        if not return_aux:
            return logits

        aux = dict(motif_aux)
        aux["pixel_logits"] = self.pixel_aux_head(pixel_vector)
        aux["motif_logits"] = self.motif_aux_head(motif_vector)

        extra = {
            "motif_position": motif_position,
            "motif_type": motif_type,
        }
        return logits, aux, extra

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model_probe = MPGFER(GRAPH_BUFFERS, TRAIN_MEAN, TRAIN_STD)
print("trainable parameters:", f"{count_parameters(model_probe):,}")
del model_probe


## 10. Loss, Metrics, Checkpoint and Scheduler Utilities

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def compute_training_loss(logits, aux, labels):
    main_ce = criterion(logits, labels)
    motif_ce = criterion(aux["motif_logits"], labels)
    pixel_ce = criterion(aux["pixel_logits"], labels)

    total = (
        main_ce
        + MOTIF_CE_WEIGHT * motif_ce
        + PIXEL_CE_WEIGHT * pixel_ce
        + MOTIF_DIVERSITY_WEIGHT * aux["motif_diversity"]
        + MOTIF_BALANCE_WEIGHT * aux["motif_balance"]
    )
    parts = {
        "main_ce": float(main_ce.detach().cpu()),
        "motif_ce": float(motif_ce.detach().cpu()),
        "pixel_ce": float(pixel_ce.detach().cpu()),
        "motif_diversity": float(aux["motif_diversity"].detach().cpu()),
        "motif_balance": float(aux["motif_balance"].detach().cpu()),
    }
    return total, parts

def metrics_from_arrays(labels, predictions):
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro")),
    }

@torch.inference_mode()
def evaluate(model, loader, use_tta=False):
    model.eval()
    all_labels = []
    all_predictions = []
    total_loss = 0.0
    total_count = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits = model(images)
            if use_tta:
                flip_logits = model(torch.flip(images, dims=[3]))
                logits = 0.5 * (logits + flip_logits)
            loss = criterion(logits, labels)

        predictions = logits.argmax(dim=-1)
        batch = labels.numel()
        total_loss += float(loss.detach().cpu()) * batch
        total_count += batch
        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

    metrics = metrics_from_arrays(all_labels, all_predictions)
    metrics["loss"] = total_loss / max(1, total_count)
    return metrics, np.asarray(all_labels), np.asarray(all_predictions)

def build_scheduler(optimizer, updates_per_epoch):
    total_steps = max(1, MAX_EPOCHS * updates_per_epoch)
    warmup_steps = max(1, WARMUP_EPOCHS * updates_per_epoch)

    def lr_lambda(step):
        if step < warmup_steps:
            return max(1e-3, (step + 1) / warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
        return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

def checkpoint_is_better(candidate, best):
    if best is None:
        return True
    if candidate["accuracy"] > best["accuracy"] + 1e-12:
        return True
    if abs(candidate["accuracy"] - best["accuracy"]) <= 1e-12:
        if candidate["macro_f1"] > best["macro_f1"] + 1e-12:
            return True
        if abs(candidate["macro_f1"] - best["macro_f1"]) <= 1e-12:
            return candidate["loss"] < best["loss"]
    return False


## 11. Mandatory Preflight

This cell is deliberately strict because the full experiment is intended to run once.

It verifies:

- forward shapes (`[B,7]`, 49 motif nodes);
- finite logits/loss/gradients;
- checkpoint round-trip;
- a fresh tiny model can memorize a small non-augmented subset.

The micro-overfit model is discarded afterward. The full experiment is instantiated fresh with the same seed.


In [ ]:
import copy
import gc

def one_step_smoke_test():
    seed_everything(SEED)
    model = MPGFER(GRAPH_BUFFERS, TRAIN_MEAN, TRAIN_STD).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)

    sample_images = torch.from_numpy(train_images[:2]).float().unsqueeze(1).to(DEVICE) / 255.0
    sample_labels = torch.from_numpy(train_labels[:2]).long().to(DEVICE)

    model.train()
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
        logits, aux, extra = model(sample_images, return_aux=True)
        loss, _ = compute_training_loss(logits, aux, sample_labels)

    if logits.shape != (2, NUM_CLASSES):
        raise RuntimeError(f"Bad logits shape: {logits.shape}")
    if extra["motif_position"].shape != (2, 49, 2):
        raise RuntimeError(f"Bad motif position shape: {extra['motif_position'].shape}")
    if extra["motif_type"].shape != (2, 49, MOTIF_TYPES):
        raise RuntimeError(f"Bad motif type shape: {extra['motif_type'].shape}")
    if not torch.isfinite(logits).all() or not torch.isfinite(loss):
        raise RuntimeError("Non-finite forward/loss in smoke test")

    loss.backward()
    finite_grads = True
    grad_seen = False
    for p in model.parameters():
        if p.grad is not None:
            grad_seen = True
            finite_grads = finite_grads and bool(torch.isfinite(p.grad).all())
    if not grad_seen or not finite_grads:
        raise RuntimeError("Missing or non-finite gradient in smoke test")

    optimizer.step()

    # State-dict roundtrip.
    temp_ckpt = OUTPUT_ROOT / "_preflight_roundtrip.pt"
    torch.save(model.state_dict(), temp_ckpt)
    clone = MPGFER(GRAPH_BUFFERS, TRAIN_MEAN, TRAIN_STD).to(DEVICE)
    clone.load_state_dict(torch.load(temp_ckpt, map_location=DEVICE, weights_only=True))
    temp_ckpt.unlink()

    params = count_parameters(model)
    print("one-step smoke: PASS")
    print("parameters:", f"{params:,}")
    print("peak GPU MB:", round(torch.cuda.max_memory_allocated() / 2**20, 1))

    del model, clone, optimizer, sample_images, sample_labels
    gc.collect()
    torch.cuda.empty_cache()

one_step_smoke_test()

def micro_overfit_check():
    if not RUN_MICRO_OVERFIT:
        print("micro-overfit skipped by configuration")
        return

    seed_everything(SEED + 101)
    n = 16
    images = torch.from_numpy(train_images[:n]).float().unsqueeze(1).to(DEVICE) / 255.0
    labels = torch.from_numpy(train_labels[:n]).long().to(DEVICE)

    model = MPGFER(GRAPH_BUFFERS, TRAIN_MEAN, TRAIN_STD).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0)
    scaler = torch.cuda.amp.GradScaler(enabled=True)

    model.train()
    final_acc = 0.0
    for step in range(80):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits, aux, _ = model(images, return_aux=True)
            loss, _ = compute_training_loss(logits, aux, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        final_acc = float((logits.argmax(dim=-1) == labels).float().mean().detach().cpu())
        if final_acc >= 0.9375 and step >= 20:
            break

    print(f"micro-overfit: steps={step+1}, train_acc={final_acc:.4f}")
    if final_acc < 0.875:
        raise RuntimeError(
            "Micro-overfit failed (<87.5% on 16 fixed samples). "
            "Do not launch the expensive full run."
        )

    del model, optimizer, scaler, images, labels
    gc.collect()
    torch.cuda.empty_cache()

micro_overfit_check()


## 12. Fresh Full Training Setup

In [ ]:
# Fresh experiment state after all preflight models are destroyed.
seed_everything(SEED)

train_loader = make_loader(train_dataset, BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_dataset, EVAL_BATCH_SIZE, shuffle=False)
test_loader = make_loader(test_dataset, EVAL_BATCH_SIZE, shuffle=False)

model = MPGFER(GRAPH_BUFFERS, TRAIN_MEAN, TRAIN_STD).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
scheduler = build_scheduler(optimizer, updates_per_epoch)

scaler = torch.cuda.amp.GradScaler(enabled=True)

run_config = {
    "seed": SEED,
    "max_epochs": MAX_EPOCHS,
    "min_epochs": MIN_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "base_lr": BASE_LR,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "pixel_dim": PIXEL_DIM,
    "pixel_heads": PIXEL_HEADS,
    "pixel_blocks": PIXEL_BLOCKS,
    "motif_types": MOTIF_TYPES,
    "motif_dim": MOTIF_DIM,
    "motif_heads": MOTIF_HEADS,
    "motif_blocks": MOTIF_BLOCKS,
    "motif_window": MOTIF_WINDOW,
    "motif_stride": MOTIF_STRIDE,
    "dropout": DROPOUT,
    "classifier_dropout": CLASSIFIER_DROPOUT,
    "selection_tta": USE_TTA_FOR_SELECTION,
    "private_tta": USE_TTA_FOR_PRIVATE,
    "parameters": count_parameters(model),
}
(OUTPUT_ROOT / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(json.dumps(run_config, indent=2))


## 13. One Full Train → PublicTest Selection Run

In [ ]:
history = []
best_metrics = None
best_epoch = None
epochs_without_improvement = 0
global_update = 0

def save_checkpoint(path, epoch, val_metrics):
    payload = {
        "epoch": int(epoch),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "val_metrics": val_metrics,
        "run_config": run_config,
        "train_mean": TRAIN_MEAN,
        "train_std": TRAIN_STD,
    }
    torch.save(payload, path)

if not RUN_FULL_TRAINING:
    raise RuntimeError("RUN_FULL_TRAINING must be True for the final one-shot notebook.")

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    epoch_loss = 0.0
    epoch_main_ce = 0.0
    epoch_count = 0
    train_correct = 0

    start = time.time()

    for step, (images, labels) in enumerate(train_loader, start=1):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits, aux, _ = model(images, return_aux=True)
            loss, parts = compute_training_loss(logits, aux, labels)
            scaled_loss = loss / GRAD_ACCUM_STEPS

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch={epoch}, step={step}")

        scaler.scale(scaled_loss).backward()

        should_step = (step % GRAD_ACCUM_STEPS == 0) or (step == len(train_loader))
        if should_step:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                raise RuntimeError(f"Non-finite gradient norm at epoch={epoch}, step={step}")
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_update += 1

        batch = labels.numel()
        epoch_loss += float(loss.detach().cpu()) * batch
        epoch_main_ce += parts["main_ce"] * batch
        epoch_count += batch
        train_correct += int((logits.argmax(dim=-1) == labels).sum().detach().cpu())

    train_metrics = {
        "loss": epoch_loss / epoch_count,
        "main_ce": epoch_main_ce / epoch_count,
        "accuracy": train_correct / epoch_count,
    }

    val_metrics, _, _ = evaluate(
        model,
        val_loader,
        use_tta=USE_TTA_FOR_SELECTION,
    )

    record = {
        "epoch": epoch,
        "lr": optimizer.param_groups[0]["lr"],
        "train": train_metrics,
        "val": val_metrics,
        "elapsed_sec": time.time() - start,
        "gpu_peak_mb": torch.cuda.max_memory_allocated() / 2**20,
    }
    history.append(record)

    print(
        f"epoch {epoch:03d} | "
        f"train acc {train_metrics['accuracy']:.4f} loss {train_metrics['loss']:.4f} | "
        f"val acc {val_metrics['accuracy']:.4f} f1 {val_metrics['macro_f1']:.4f} "
        f"loss {val_metrics['loss']:.4f} | "
        f"lr {record['lr']:.2e}"
    )

    if checkpoint_is_better(val_metrics, best_metrics):
        best_metrics = dict(val_metrics)
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(CHECKPOINT_PATH, epoch, val_metrics)
        print("  -> new best checkpoint")
    else:
        epochs_without_improvement += 1

    # Save last only for forensic recovery. The scientific selection remains best_val_acc.
    save_checkpoint(LAST_PATH, epoch, val_metrics)
    (OUTPUT_ROOT / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

    if epoch >= MIN_EPOCHS and epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(
            f"Early stop: no PublicTest improvement for {EARLY_STOP_PATIENCE} epochs "
            f"after minimum epoch requirement."
        )
        break

if not CHECKPOINT_PATH.is_file():
    raise RuntimeError("Training ended without a best checkpoint.")

print("best epoch:", best_epoch)
print("best PublicTest selection metrics:", best_metrics)


## 14. Freeze Best Checkpoint → Final PublicTest and PrivateTest

PrivateTest is touched only here, after checkpoint selection is complete.

Both raw and the predeclared 2-view TTA metrics are saved. The TTA metric is the primary final inference protocol because it was declared before training.


In [ ]:
best_payload = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_payload["model_state"], strict=True)
model.eval()

val_raw, val_y, val_pred_raw = evaluate(model, val_loader, use_tta=False)
val_tta, _, val_pred_tta = evaluate(model, val_loader, use_tta=True)

test_raw, test_y, test_pred_raw = evaluate(model, test_loader, use_tta=False)
test_tta, _, test_pred_tta = evaluate(model, test_loader, use_tta=USE_TTA_FOR_PRIVATE)

final_metrics = {
    "best_epoch": int(best_payload["epoch"]),
    "selection_protocol": "PublicTest 2-view TTA" if USE_TTA_FOR_SELECTION else "PublicTest raw",
    "private_protocol": "PrivateTest 2-view TTA" if USE_TTA_FOR_PRIVATE else "PrivateTest raw",
    "public_raw": val_raw,
    "public_tta": val_tta,
    "private_raw": test_raw,
    "private_tta": test_tta,
}

print(json.dumps(final_metrics, indent=2))
(OUTPUT_ROOT / "final_metrics.json").write_text(json.dumps(final_metrics, indent=2), encoding="utf-8")

private_final_pred = test_pred_tta if USE_TTA_FOR_PRIVATE else test_pred_raw
report = classification_report(
    test_y,
    private_final_pred,
    labels=list(range(NUM_CLASSES)),
    output_dict=True,
    zero_division=0,
)
(OUTPUT_ROOT / "private_classification_report.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8",
)

cm = confusion_matrix(test_y, private_final_pred, labels=list(range(NUM_CLASSES)))
np.save(OUTPUT_ROOT / "private_confusion_matrix.npy", cm)


## 15. Post-run Diagnostics

In [ ]:
import matplotlib.pyplot as plt

# Learning curves
epochs = [r["epoch"] for r in history]
train_acc = [r["train"]["accuracy"] for r in history]
val_acc = [r["val"]["accuracy"] for r in history]
val_f1 = [r["val"]["macro_f1"] for r in history]

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_acc, label="Train accuracy")
plt.plot(epochs, val_acc, label="PublicTest accuracy")
plt.plot(epochs, val_f1, label="PublicTest macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.title("MPG-FER training history")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "training_curves.png", dpi=160)
plt.show()

# Confusion matrix
plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.title("PrivateTest confusion matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.xticks(range(NUM_CLASSES))
plt.yticks(range(NUM_CLASSES))
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "private_confusion_matrix.png", dpi=160)
plt.show()

# Motif diagnostics on a bounded PublicTest sample.
diag_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
diag_images, diag_labels = next(iter(diag_loader))
diag_images = diag_images.to(DEVICE)
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
    _, aux, extra = model(diag_images, return_aux=True)

motif_type = extra["motif_type"].float().cpu().numpy()      # [B,49,M]
motif_position = extra["motif_position"].float().cpu().numpy()
usage = motif_type.mean(axis=(0, 1))
motif_diag = {
    "bounded_sample_size": int(diag_images.shape[0]),
    "usage_min": float(usage.min()),
    "usage_max": float(usage.max()),
    "usage_entropy": float(-(usage * np.log(usage + 1e-12)).sum()),
    "top_types": np.argsort(-usage)[:10].astype(int).tolist(),
}
print("motif diagnostics:", motif_diag)
(OUTPUT_ROOT / "motif_diagnostics.json").write_text(
    json.dumps(motif_diag, indent=2),
    encoding="utf-8",
)


## 16. Validate and Archive All Outputs

In [ ]:
required_outputs = [
    CHECKPOINT_PATH,
    OUTPUT_ROOT / "run_config.json",
    OUTPUT_ROOT / "environment.json",
    OUTPUT_ROOT / "dataset_manifest.json",
    OUTPUT_ROOT / "history.json",
    OUTPUT_ROOT / "final_metrics.json",
    OUTPUT_ROOT / "private_classification_report.json",
    OUTPUT_ROOT / "private_confusion_matrix.npy",
    OUTPUT_ROOT / "training_curves.png",
    OUTPUT_ROOT / "private_confusion_matrix.png",
    OUTPUT_ROOT / "motif_diagnostics.json",
]
missing = [str(p) for p in required_outputs if not p.is_file()]
if missing:
    raise RuntimeError("Missing required outputs:\n" + "\n".join(missing))

completion = {
    "status": "complete",
    "best_checkpoint": str(CHECKPOINT_PATH),
    "best_epoch": int(best_payload["epoch"]),
    "final_metrics": final_metrics,
    "required_outputs": [str(p) for p in required_outputs],
}
(OUTPUT_ROOT / "TRAINING_COMPLETE.json").write_text(
    json.dumps(completion, indent=2),
    encoding="utf-8",
)

if ARCHIVE_PATH.exists():
    ARCHIVE_PATH.unlink()

shutil.make_archive(
    str(ARCHIVE_PATH.with_suffix("")),
    "zip",
    root_dir=OUTPUT_ROOT.parent,
    base_dir=OUTPUT_ROOT.name,
)

print("TRAINING_COMPLETE:", OUTPUT_ROOT / "TRAINING_COMPLETE.json")
print("archive:", ARCHIVE_PATH)
print("archive size MiB:", round(ARCHIVE_PATH.stat().st_size / 2**20, 2))
